# Day 7 | 백테스팅
## Korean Stock Undervaluation Analysis — Backtesting

**목표:**
- 2025-03-30 기준 괴리율 재계산
- 3m / 6m / 9m / 12m 수익률 측정 (옵션B: 월말 / 옵션C: 만기+5영업일)
- 벤치마크 대비 알파 계산
- 가설검정 (Pearson / Spearman / ANOVA)

**핵심 설계:**
```
기준일: 2025-03-30 (옵션B) / 2025-03-20 (옵션C)
EPS: 2024 연간 재무제표 (당시 투자자가 사용 가능한 최신 데이터)
측정: 3m / 6m / 9m / 12m 수익률
벤치마크: KOSPI / KOSDAQ / KRX가중(85:15) / 기준금리 / CD91일 / 미국10년물
```

**가설:**
```
H0: 괴리율과 수익률 간 상관관계 없음
H1: 괴리율이 낮을수록 수익률이 높다
```

**결과 요약:**
```
Pearson r = -0.156 (12M) → H0 기각 불가 (p=0.070)
ANOVA p < 0.0001 → 시그널 그룹 간 수익률 차이 유의
강력매수 12M: +201.1% vs KOSPI +112.6% → 알파 +88.5%
```


## 0. 환경 세팅

In [ ]:
!pip install -q git+https://github.com/FinanceData/FinanceDataReader.git

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import FinanceDataReader as fdr
import requests
import time
import os
from datetime import datetime, timedelta
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

BASE_PATH = '/content/drive/MyDrive/stock-analysis'

df_v2 = pd.read_csv(
    f'{BASE_PATH}/data/output/gap_v2.csv',
    dtype={'Code': str, 'corp_code': str}
)
df_v2['corp_code'] = df_v2['corp_code'].str.zfill(8)

# 기준일 + 측정 시점 설정
BASE_DATE = '2025-03-30'
DATES_B = {'base': '2025-03-30', '3m': '2025-06-30',
           '6m': '2025-09-30', '9m': '2025-12-30', '12m': '2026-03-28'}
DATES_C = {'base': '2025-03-20', '3m': '2025-06-19',
           '6m': '2025-09-18', '9m': '2025-12-18', '12m': '2026-03-20'}

print(f"✅ df_v2: {len(df_v2)}개")
print(f"기준일(B): {DATES_B['base']} / 기준일(C): {DATES_C['base']}")


## 1. 주가 수집 (전체 기간 한 번에)

In [ ]:
# 2025-03-15 ~ 2026-03-31 전체 기간 한 번에 수집
# → 각 시점 날짜 인덱싱으로 B/C 옵션 모두 커버

prices_all = {}
failed_all = []

for idx, row in df_v2.iterrows():
    code = row['Code']
    try:
        df_p = fdr.DataReader(code, '2025-03-15', '2026-03-31')
        if len(df_p) > 0:
            df_p.index = pd.to_datetime(df_p.index)
            prices_all[code] = df_p['Close']
        else:
            failed_all.append(code)
    except:
        failed_all.append(code)
    time.sleep(0.05)

    if (idx + 1) % 30 == 0 or (idx + 1) == 136:
        print(f"[{idx+1:3d}/136] 수집: {len(prices_all)}개 | 실패: {len(failed_all)}개")

print(f"✅ 수집 완료: {len(prices_all)}개")

def get_price_on_date(series, target_date):
    target    = pd.Timestamp(target_date)
    available = series.index[series.index <= target]
    return series.loc[available[-1]] if len(available) > 0 else None


## 2. 괴리율 재계산 + 수익률 계산

In [ ]:
# 2025-03-30 주가 + 2024 EPS → PER 재계산
# → "1년 전 이 시점에 분석했다면?" 재현

results = []
for _, row in df_v2.iterrows():
    code = row['Code']
    eps  = row['EPS']
    if pd.isna(eps) or eps <= 0 or code not in prices_all:
        continue

    p = prices_all[code]
    price_B_base = get_price_on_date(p, DATES_B['base'])
    price_C_base = get_price_on_date(p, DATES_C['base'])
    if not price_B_base or not price_C_base:
        continue

    entry = {
        'Code': code, 'Name': row['Name'],
        'Sector': row['Sector'], 'EPS': eps,
        'signal_v2': row['signal_v2'],
        'price_B_base': price_B_base,
        'price_C_base': price_C_base,
    }
    for opt, dates in [('B', DATES_B), ('C', DATES_C)]:
        for period in ['3m', '6m', '9m', '12m']:
            entry[f'price_{opt}_{period}'] = get_price_on_date(p, dates[period])
    results.append(entry)

df_bt = pd.DataFrame(results)

# PER + 괴리율 재계산
df_bt['per_B'] = df_bt['price_B_base'] / df_bt['EPS']
df_bt['per_C'] = df_bt['price_C_base'] / df_bt['EPS']

market_per_B = df_bt['per_B'].median()
sector_per_B = df_bt.groupby('Sector')['per_B'].median()
df_bt['sector_per_B'] = df_bt['Sector'].map(sector_per_B)

df_bt['gap_market_B'] = (df_bt['per_B'] - market_per_B) / market_per_B
df_bt['gap_sector_B'] = (df_bt['per_B'] - df_bt['sector_per_B']) / df_bt['sector_per_B']

# 수익률 계산
for period in ['3m', '6m', '9m', '12m']:
    for opt in ['B', 'C']:
        df_bt[f'ret_{opt}_{period}'] = (
            df_bt[f'price_{opt}_{period}'] - df_bt[f'price_{opt}_base']
        ) / df_bt[f'price_{opt}_base'] * 100

print(f"✅ 백테스팅 데이터 구성: {len(df_bt)}개")
print(f"   시장 중앙값 PER (B): {market_per_B:.2f}")
print(f"   평균 12M 수익률 (B): {df_bt['ret_B_12m'].mean():.1f}%")


## 3. 가설검정 + ANOVA

In [ ]:
# Pearson / Spearman 상관분석
results_stats = []
for opt in ['B', 'C']:
    for period in ['3m', '6m', '9m', '12m']:
        df_clean = df_bt[[f'gap_market_{opt}', f'ret_{opt}_{period}']].dropna()
        x, y = df_clean.values[:, 0], df_clean.values[:, 1]
        pr, pp = stats.pearsonr(x, y)
        sr, sp = stats.spearmanr(x, y)
        results_stats.append({
            'option': opt, 'period': period,
            'pearson_r': round(pr, 3), 'pearson_p': round(pp, 4),
            'spearman_r': round(sr, 3), 'spearman_p': round(sp, 4),
        })

df_stats = pd.DataFrame(results_stats)
print("── Pearson 상관계수 ──")
print(df_stats[df_stats['option']=='B'][['period','pearson_r','pearson_p']].to_string(index=False))

# ANOVA
print("\n── ANOVA (시그널 그룹별 수익률 차이) ──")
signal_order = ['강력매수', '매수', '중립', '매도', '강력매도']
for period in ['3m', '6m', '9m', '12m']:
    groups = [
        df_bt[df_bt['signal_v2']==s][f'ret_B_{period}'].dropna().values
        for s in signal_order if len(df_bt[df_bt['signal_v2']==s]) >= 3
    ]
    f_stat, p_val = stats.f_oneway(*groups)
    sig = "✅ 유의" if p_val < 0.05 else "❌"
    print(f"  {period}: F={f_stat:.3f} p={p_val:.4f} {sig}")


## 4. 저장

In [ ]:
df_bt.to_csv(f'{BASE_PATH}/data/output/backtest_result.csv',
             index=False, encoding='utf-8-sig')
df_stats.to_csv(f'{BASE_PATH}/data/output/backtest_stats.csv',
                index=False, encoding='utf-8-sig')
print("✅ backtest_result.csv 저장 완료")
print("✅ backtest_stats.csv 저장 완료")
